In [ ]:
os.environ["OPENROUTER_API_KEY"] = "x"
os.environ["TAVILY_API_KEY"] = "x"

In [20]:
import os
import requests
from bs4 import BeautifulSoup

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from tavily import TavilyClient


tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def internet_search(query: str) -> dict:
    """Search the internet and return the top 3 ranked search results."""
    return tavily_client.search(
        query=query,
        max_results=3
    )

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL."""
    response = requests.get(
        url,
        timeout=10.0,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "header", "footer", "svg"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    cleaned_text = "\n".join(line for line in lines if line)

    return cleaned_text[:8000]

llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

system_prompt = """
You are a web research agent.

Your task is to answer the user's question using the content from the top 3 ranking sites.

Instructions:
1. Optionally refine the user's question before searching.
2. Call internet_search first.
3. Use only the top 3 search results returned by internet_search.
4. Extract the URLs from those 3 results.
5. Call fetch_url on each of those 3 URLs.
6. Read and compare the content from those pages.
7. Answer the user's question using only information supported by those fetched pages.
8. Do not invent facts.
9. If the sources disagree, mention that clearly.
10. If the pages do not provide enough information, say so clearly.
11. End with a short Sources section listing the 3 URLs used.
"""

agent = create_agent(
    model=llm,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt
)

user_question = "What is agentic AI and how is it different from normal software?"

result = agent.invoke({
    "messages": [
        {"role": "user", "content": user_question}
    ]
})

print(result["messages"][-1].content)

**Agentic AI** is a class of artificial‑intelligence systems that possess **autonomous, goal‑directed intelligence**.  These systems can *plan, make decisions, and take actions* to achieve specified objectives with little or no human intervention.  The emphasis is on the **capability** of exhibiting independent, adaptive behavior rather than on a particular technical implementation [1](https://www.matillion.com/blog/agentic-ai-vs-ai-agents).

**Normal (traditional) software / traditional AI** differs in that it typically:
- Operates on **predefined rules or explicit prompts**,
- Does **not autonomously plan or adapt** to new conditions,
- Requires **direct human instruction** for each step [2](https://www.fullstack.com/labs/resources/blog/agentic-ai-vs-traditional-ai-what-sets-ai-agents-apart).

Instead of being a static program, **AI agents** are the concrete software implementations that embody the agentic‑AI capability, orchestrating APIs, workflows, and other tools to carry out mul